# India Climate & Energy Atlas
## A reproducible exploration of development and the energy transition

**Data source:** World Bank Indicators API (live retrieval).  
**Countries:** India, Bangladesh, China, Indonesia, Sri Lanka and Vietnam.  
**Research focus:** GDP per capita, electricity access, renewable energy consumption, CO₂ emissions, population and electricity consumption.

This notebook uses real observations returned by the API. It does not create synthetic data. **Enable Internet in the notebook environment** so the API requests can run. If Internet is unavailable, the notebook stops with an explanatory error rather than silently substituting fabricated values.

**Important:** This is observational, country-level analysis. Correlations are not causal effects. Indicator coverage and publication lags differ by series.

## 1. Retrieve the real data

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display

BASE_URL = "https://api.worldbank.org/v2"
START_YEAR, END_YEAR = 2000, 2023
COUNTRIES = {
    "IND": "India", "BGD": "Bangladesh", "CHN": "China",
    "IDN": "Indonesia", "LKA": "Sri Lanka", "VNM": "Vietnam"
}
INDICATORS = {
    "NY.GDP.PCAP.KD": "GDP per capita (constant 2015 US$)",
    "EN.ATM.CO2E.PC": "CO2 emissions (metric tons per capita)",
    "EG.ELC.ACCS.ZS": "Access to electricity (% of population)",
    "EG.FEC.RNEW.ZS": "Renewable energy consumption (% of total final energy consumption)",
    "SP.POP.TOTL": "Population, total",
    "EG.USE.ELEC.KH.PC": "Electric power consumption (kWh per capita)"
}

def fetch_indicator(indicator, retries=3):
    country_path = ";".join(COUNTRIES)
    url = f"{BASE_URL}/country/{country_path}/indicator/{indicator}"
    params = {"date": f"{START_YEAR}:{END_YEAR}", "format": "json", "per_page": 20000}
    last_error = None
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=45)
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, list) or len(payload) < 2 or not payload[1]:
                return pd.DataFrame(columns=["country_code", "country", "year", "indicator_code", "indicator", "value"])
            rows = []
            for item in payload[1]:
                rows.append({
                    "country_code": item.get("countryiso3code"),
                    "country": (item.get("country") or {}).get("value"),
                    "year": int(item["date"]),
                    "indicator_code": indicator,
                    "indicator": INDICATORS[indicator],
                    "value": item.get("value")
                })
            return pd.DataFrame(rows)
        except Exception as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"Could not retrieve {indicator}: {last_error}") from last_error

try:
    long_data = pd.concat([fetch_indicator(code) for code in INDICATORS], ignore_index=True)
except Exception as exc:
    raise RuntimeError(
        "Live World Bank data could not be downloaded. Enable Internet for this notebook and rerun. "
        "No synthetic fallback is used."
    ) from exc

if long_data.empty:
    raise RuntimeError("The API returned no rows. Check the API and rerun later.")

long_data["value"] = pd.to_numeric(long_data["value"], errors="coerce")
data = long_data.pivot_table(
    index=["country_code", "country", "year"],
    columns="indicator_code", values="value", aggfunc="first"
).reset_index()
data.columns.name = None
data = data.rename(columns=INDICATORS).sort_values(["country_code", "year"]).reset_index(drop=True)
data = data.drop_duplicates(["country_code", "year"])
print(f"Retrieved {len(data):,} country-year rows for {data.country_code.nunique()} countries.")
print(f"Observed year range: {int(data.year.min())}–{int(data.year.max())}")
print("Retrieved at UTC:", datetime.now(timezone.utc).isoformat())
display(data.head())

## 2. Audit coverage and missingness

In [ ]:
identifier_cols = {"country_code", "country", "year"}
metric_cols = [c for c in data.columns if c not in identifier_cols]
coverage = pd.DataFrame([
    {
        "indicator": col,
        "non_missing": int(data[col].notna().sum()),
        "missing": int(data[col].isna().sum()),
        "coverage_pct": round(float(data[col].notna().mean() * 100), 2)
    }
    for col in metric_cols
]).sort_values("coverage_pct", ascending=False)
display(coverage)
ax = coverage.sort_values("coverage_pct").plot(
    kind="barh", x="indicator", y="coverage_pct", legend=False, figsize=(10, 5),
    title="Indicator coverage across the country-year panel"
)
ax.set_xlabel("Non-missing observations (%)")
ax.set_ylabel("")
ax.set_xlim(0, 100)
ax.grid(axis="x", alpha=.25)
plt.tight_layout()
plt.show()
print("Missing values remain missing; they are never silently replaced with zero.")

## 3. India trends

In [ ]:
GDP = "GDP per capita (constant 2015 US$)"
CO2 = "CO2 emissions (metric tons per capita)"
ELECTRICITY = "Access to electricity (% of population)"
RENEWABLES = "Renewable energy consumption (% of total final energy consumption)"
core_metrics = [GDP, CO2, ELECTRICITY, RENEWABLES]
india = data[data.country_code == "IND"].sort_values("year").copy()

for metric in core_metrics:
    if metric in india.columns and india[metric].notna().sum() >= 2:
        observed = india[["year", metric]].dropna()
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(observed.year, observed[metric], marker="o", linewidth=2)
        ax.set_title(metric)
        ax.set_xlabel("Year")
        ax.set_ylabel(metric)
        ax.grid(True, alpha=.25)
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough observed data for:", metric)

## 4. Endpoint changes with actual observation years

In [ ]:
trend_rows = []
for metric in core_metrics:
    if metric not in india.columns:
        continue
    obs = india[["year", metric]].dropna().sort_values("year")
    if obs.empty:
        continue
    first, last = obs.iloc[0], obs.iloc[-1]
    pct = ((last[metric] / first[metric]) - 1) * 100 if first[metric] != 0 else np.nan
    trend_rows.append({
        "indicator": metric,
        "first_year": int(first.year), "first_value": float(first[metric]),
        "last_year": int(last.year), "last_value": float(last[metric]),
        "endpoint_change_pct": float(pct) if pd.notna(pct) else None,
        "observations": int(len(obs))
    })
trend_table = pd.DataFrame(trend_rows)
display(trend_table)
print("Endpoint change compares first and last observed values, which may span different years by indicator.")

## 5. GDP and CO₂: complete-pair sensitivity check

In [ ]:
paired = india[["year", GDP, CO2]].dropna().sort_values("year")
print("Complete years with both indicators:", len(paired))
if len(paired) >= 3:
    pearson_r = paired[GDP].corr(paired[CO2], method="pearson")
    spearman_r = paired[GDP].corr(paired[CO2], method="spearman")
    print(f"Pearson correlation: {pearson_r:.3f}")
    print(f"Spearman correlation: {spearman_r:.3f}")
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(paired[GDP], paired[CO2])
    for _, row in paired.iterrows():
        ax.annotate(str(int(row.year)), (row[GDP], row[CO2]), fontsize=7, alpha=.7)
    ax.set_xlabel(GDP)
    ax.set_ylabel(CO2)
    ax.set_title("India: GDP per capita vs CO₂ per capita")
    ax.grid(True, alpha=.25)
    plt.tight_layout()
    plt.show()
else:
    pearson_r = spearman_r = None
    print("Insufficient complete paired observations for correlation.")
print("Interpretation: descriptive association only. Trends, omitted variables and measurement differences can confound correlation.")

## 6. Country comparison over observed data

In [ ]:
selected = data[data.country_code.isin(COUNTRIES)].copy()
for metric in [GDP, CO2, ELECTRICITY, RENEWABLES]:
    if metric not in selected.columns:
        continue
    fig, ax = plt.subplots(figsize=(10, 5))
    for code, group in selected.groupby("country_code"):
        obs = group[["year", metric]].dropna().sort_values("year")
        if not obs.empty:
            ax.plot(obs.year, obs[metric], label=COUNTRIES.get(code, code), linewidth=1.8)
    ax.set_title(metric)
    ax.set_xlabel("Year")
    ax.set_ylabel(metric)
    ax.legend(ncol=2, frameon=False)
    ax.grid(True, alpha=.25)
    plt.tight_layout()
    plt.show()

if GDP in selected and CO2 in selected:
    display(selected.dropna(subset=[GDP, CO2]).groupby(["country_code", "country"]).agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        complete_years=("year", "size"),
        mean_gdp_per_capita=(GDP, "mean"),
        mean_co2_per_capita=(CO2, "mean")
    ).reset_index())

## 7. Export reproducible artifacts

In [ ]:
OUT = Path("earth_atlas_outputs")
OUT.mkdir(exist_ok=True)
data.to_csv(OUT / "world_bank_panel.csv", index=False)
coverage.to_csv(OUT / "coverage.csv", index=False)
metadata = {
    "source": "World Bank Indicators API",
    "source_url": BASE_URL,
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "start_year": START_YEAR,
    "end_year": END_YEAR,
    "countries": COUNTRIES,
    "indicators": INDICATORS,
    "rows": int(len(data)),
    "missing_values_preserved": True,
    "interpretation_note": "Descriptive analysis only; correlations do not establish causation."
}
(OUT / "data_provenance.json").write_text(json.dumps(metadata, indent=2))
print("Saved real panel, coverage table and provenance metadata to:", OUT.resolve())
display(data.tail())

## 8. Limitations and publication checklist

- The dataset is a country-level observational panel; it cannot establish causal effects.
- Indicator definitions, sources, publication lags and historical revisions can differ.
- The World Bank electricity-access series may include modeled estimates; consult its metadata before interpreting changes.
- Renewable energy consumption is not the same as renewable electricity generation.
- Do not claim “emissions decoupling” from a simple correlation. Define the term, distinguish absolute from per-capita emissions and test sensitivity.
- Cite the World Bank API and record the retrieval date.
- Before publishing, restart the kernel and run every cell from top to bottom.
- Never claim a result or accuracy score that is not reproduced by the notebook.